# 09 Sampling Optimization

对比未采样、普通 SMOTE-ENN、优化后的 SMOTENC+ENN 三组训练方案，并输出最优采样配置。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for candidate in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from src.features.feature_selector import CreditFeatureSelector
from src.features.preprocessor import CreditDataPreprocessor
from src.features.sampler import CreditSampler
from src.models.model_evaluator import CreditModelEvaluator
from src.models.risk_classifier import CreditRiskClassifier

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "src" / "models"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["正常类", "关注类", "次级类", "可疑类", "损失类"]


In [ ]:
def load_best_weight_configuration() -> tuple[str, dict[int, float] | None]:
    config_path = MODEL_DIR / "best_class_weight_config.json"
    if not config_path.exists():
        return "none", None

    payload = json.loads(config_path.read_text(encoding="utf-8"))
    weight_mode = str(payload.get("class_weight_mode", "none"))
    resolved_weight = payload.get("resolved_class_weight")
    if resolved_weight is None:
        return weight_mode, None

    normalized_weight = {
        int(class_label): float(class_weight)
        for class_label, class_weight in resolved_weight.items()
    }
    return weight_mode, normalized_weight


best_weight_mode, best_weight_dict = load_best_weight_configuration()
display(Markdown(f"当前默认采用的权重模式: `{best_weight_mode}`"))
display(Markdown(f"当前默认采用的权重字典: `{best_weight_dict}`"))


In [ ]:
train_df = pd.read_csv(resolve_split_path("train"), low_memory=False)
val_df = pd.read_csv(resolve_split_path("val"), low_memory=False)
test_df = pd.read_csv(resolve_split_path("test"), low_memory=False)

y_train = train_df["preloan_risk_label"].astype(int)
y_val = val_df["preloan_risk_label"].astype(int)
y_test = test_df["preloan_risk_label"].astype(int)

preprocessor = CreditDataPreprocessor(target_column="preloan_risk_label")
X_train_processed = preprocessor.fit_transform(train_df)
X_val_processed = preprocessor.transform(val_df)
X_test_processed = preprocessor.transform(test_df)

categorical_feature_indices = [
    index
    for index, column in enumerate(X_train_processed.columns)
    if "=" in column and not column.endswith("__target_encoded")
]

display(Markdown(f"识别出的分类特征列索引数量: `{len(categorical_feature_indices)}`"))


In [ ]:
def build_plain_smoteenn(X_train: pd.DataFrame, y_train: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    label_counts = y_train.value_counts().to_dict()
    majority_count = max(label_counts.values())
    eligible_classes = {
        int(label): int(majority_count)
        for label, count in label_counts.items()
        if 1 < count < majority_count
    }
    minimum_count = min((count for count in label_counts.values() if count > 1), default=0)

    if not eligible_classes or minimum_count < 2:
        return X_train.copy(), y_train.copy()

    smoteenn = SMOTEENN(
        random_state=42,
        sampling_strategy=eligible_classes,
        smote=SMOTE(
            random_state=42,
            k_neighbors=max(1, min(5, minimum_count - 1)),
            sampling_strategy=eligible_classes,
        ),
    )
    X_resampled, y_resampled = smoteenn.fit_resample(X_train, y_train)
    return (
        pd.DataFrame(X_resampled, columns=X_train.columns).astype(np.float32),
        pd.Series(y_resampled, name="target"),
    )


def compute_high_risk_recall(metric_payload: dict) -> float:
    report = metric_payload["classification_report_named"]
    risk_recalls = [
        float(report.get(class_name, {}).get("recall", 0.0))
        for class_name in metric_payload["class_names"]
        if class_name != "正常类"
    ]
    if not risk_recalls:
        return 0.0
    return float(np.mean(risk_recalls))


def evaluate_model_bundle(experiment_name: str, X_train_bundle: pd.DataFrame, y_train_bundle: pd.Series):
    selector = CreditFeatureSelector(top_k_features=80, use_pca=False, n_estimators=100)
    X_train_selected = selector.fit_transform(X_train_bundle, y_train_bundle)
    X_val_selected = selector.transform(X_val_processed)
    X_test_selected = selector.transform(X_test_processed)

    classifier = CreditRiskClassifier(
        model_type="lightgbm",
        class_weight_mode=best_weight_mode,
        custom_class_weight=best_weight_dict,
    )
    classifier.fit(X_train_selected, y_train_bundle)

    train_metrics = CreditModelEvaluator(
        classifier,
        X_train_selected,
        y_train_bundle,
        CLASS_NAMES,
    ).evaluate_imbalanced_multiclass()
    val_metrics = CreditModelEvaluator(
        classifier,
        X_val_selected,
        y_val,
        CLASS_NAMES,
    ).evaluate_imbalanced_multiclass()
    test_metrics = CreditModelEvaluator(
        classifier,
        X_test_selected,
        y_test,
        CLASS_NAMES,
    ).evaluate_imbalanced_multiclass()

    return {
        "experiment_name": experiment_name,
        "train_distribution": dict(sorted(pd.Series(y_train_bundle).value_counts().to_dict().items())),
        "train_macro_f1": float(train_metrics["macro_f1"]),
        "val_macro_f1": float(val_metrics["macro_f1"]),
        "test_macro_f1": float(test_metrics["macro_f1"]),
        "val_weighted_f1": float(val_metrics["weighted_f1"]),
        "test_weighted_f1": float(test_metrics["weighted_f1"]),
        "val_high_risk_recall": compute_high_risk_recall(val_metrics),
        "test_high_risk_recall": compute_high_risk_recall(test_metrics),
        "macro_f1_overfit_gap": float(train_metrics["macro_f1"] - test_metrics["macro_f1"]),
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


In [ ]:
optimized_sampler = CreditSampler(categorical_features=categorical_feature_indices)
X_train_opt, y_train_opt = optimized_sampler.fit_resample(
    X_train_processed,
    y_train,
    sampling_strategy="auto",
    max_oversample_ratio=5,
)
optimized_sampler.plot_sample_distribution(y_train, y_train_opt)

X_train_plain, y_train_plain = build_plain_smoteenn(X_train_processed, y_train)


In [ ]:
experiment_results = []
experiment_results.append(evaluate_model_bundle("baseline_no_sampling", X_train_processed, y_train))
experiment_results.append(evaluate_model_bundle("plain_smoteenn", X_train_plain, y_train_plain))
experiment_results.append(evaluate_model_bundle("smotenc_enn_optimized", X_train_opt, y_train_opt))

comparison_df = pd.DataFrame(
    [
        {
            "实验组": item["experiment_name"],
            "训练集 Macro-F1": round(item["train_macro_f1"], 6),
            "验证集 Macro-F1": round(item["val_macro_f1"], 6),
            "测试集 Macro-F1": round(item["test_macro_f1"], 6),
            "验证集高风险召回率": round(item["val_high_risk_recall"], 6),
            "测试集高风险召回率": round(item["test_high_risk_recall"], 6),
            "过拟合差值": round(item["macro_f1_overfit_gap"], 6),
            "训练集分布": json.dumps(item["train_distribution"], ensure_ascii=False),
        }
        for item in experiment_results
    ]
)
display(comparison_df)


In [ ]:
plot_positions = np.arange(len(comparison_df))
bar_width = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(plot_positions - bar_width, comparison_df["验证集 Macro-F1"], width=bar_width, label="验证集 Macro-F1")
ax.bar(plot_positions, comparison_df["测试集高风险召回率"], width=bar_width, label="测试集高风险召回率")
ax.bar(plot_positions + bar_width, comparison_df["过拟合差值"], width=bar_width, label="过拟合差值")
ax.set_xticks(plot_positions)
ax.set_xticklabels(comparison_df["实验组"], rotation=15, ha="right")
ax.set_ylabel("Score")
ax.set_title("采样方案效果对比")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.3)

comparison_path = FIGURE_DIR / "sampling_optimization_comparison.png"
fig.tight_layout()
fig.savefig(comparison_path, dpi=200, bbox_inches="tight")
plt.close(fig)

display(Markdown(f"效果对比图已保存到 `results/figures/{comparison_path.name}`"))


In [ ]:
best_result = sorted(
    experiment_results,
    key=lambda item: (
        item["val_macro_f1"],
        item["val_high_risk_recall"],
        -abs(item["macro_f1_overfit_gap"]),
    ),
    reverse=True,
)[0]

best_sampling_payload = {
    "selected_by": "validation_macro_f1_then_high_risk_recall_then_lower_overfit_gap",
    "experiment_name": best_result["experiment_name"],
    "class_weight_mode": best_weight_mode,
    "class_weight_config": best_weight_dict,
    "categorical_feature_count": len(categorical_feature_indices),
    "optimized_sampler_config": {
        "sampling_strategy": "auto",
        "max_oversample_ratio": 5,
    },
    "metrics": {
        "train_macro_f1": best_result["train_macro_f1"],
        "val_macro_f1": best_result["val_macro_f1"],
        "test_macro_f1": best_result["test_macro_f1"],
        "val_high_risk_recall": best_result["val_high_risk_recall"],
        "test_high_risk_recall": best_result["test_high_risk_recall"],
        "macro_f1_overfit_gap": best_result["macro_f1_overfit_gap"],
    },
    "sample_distribution_plot": "results/figures/sample_distribution_comparison.png",
    "comparison_plot": "results/figures/sampling_optimization_comparison.png",
}

best_sampling_path = MODEL_DIR / "best_sampling_config.json"
best_sampling_path.write_text(
    json.dumps(best_sampling_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(Markdown(f"最优采样配置已保存到 `src/models/{best_sampling_path.name}`"))
